In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple terms for a Robotics Engineering student."
)

llm = OllamaLLM(model="qwen2.5:3b")

chain = prompt | llm | StrOutputParser()

topics = [
    "LangChain Tools",
    "AI Agents",
    "Tool Calling",
    "Agent Memory",
    "Autonomous Robots"
]

for topic in topics:
    result = chain.invoke({"topic": topic})
    print(f"\n--- {topic} ---")
    print(result)


--- LangChain Tools ---
Certainly! Let's break down the concept of LangChain Tools in simple terms for a Robotics Engineering student.

### What are LangChain Tools?
LangChain Tools are like special helpers or tools you can use when working with programming languages. They help you with specific tasks, much like how a robot in robotics can perform specific actions.

### Why Use LangChain Tools?
Imagine you're programming a robot to do different tasks. Each task might require a different skill, like moving to a specific location, picking up objects, or cleaning a surface. Traditionally, you would need to write a lot of code for each task. This can be very time-consuming and can lead to errors if not done perfectly.

LangChain Tools are like pre-written, tested code snippets that you can use to perform specific tasks. They are designed to handle common tasks, so you don't have to write all the code from scratch for each task. This makes your programming job much easier and faster.

### 

In [2]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True)

memory_chain = PromptTemplate.from_template(
    """You are a helpful assistant.

Conversation history:
{history}

Human: {input}
Assistant:"""
) | llm | StrOutputParser()

turns = [
    "My name is Ughandhar.",
    "I am studying Robotics Engineering.",
    "What is my name?",
    "What am I studying?",
    "Can you remember both details?"
]

for i, user_input in enumerate(turns, 1):
    history = memory.load_memory_variables({})["history"]

    result = memory_chain.invoke({
        "history": history,
        "input": user_input
    })

    memory.save_context(
        {"input": user_input},
        {"output": result}
    )

    print(f"\n--- Turn {i} ---")
    print(f"Human: {user_input}")
    print(f"Assistant: {result}")

print("\n--- Final Conversation History ---")
print(memory.load_memory_variables({})["history"])

C:\Users\ughan\AppData\Local\Temp\ipykernel_11600\1217742911.py:3: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(return_messages=True)
c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)



--- Turn 1 ---
Human: My name is Ughandhar.
Assistant: It's nice to meet you, Ughandhar. How can I assist you today?

--- Turn 2 ---
Human: I am studying Robotics Engineering.
Assistant: That's great! Studying Robotics Engineering is an exciting field with many opportunities. Are you looking to learn more about specific aspects of robotics engineering, or do you have any particular questions you'd like to discuss? Whether it's about the principles of robotics, programming robots, designing robots, or any other related topics, I'm here to help.

--- Turn 3 ---
Human: What is my name?
Assistant: Your name is Ughandhar.

--- Turn 4 ---
Human: What am I studying?
Assistant: It seems like you've mentioned that you are studying Robotics Engineering in your previous conversation. Could you provide more details or ask about something specific related to your studies? For instance, do you have any questions about the principles of robotics, programming robots, designing robots, or any other re

In [3]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.tools import tool
from ddgs import DDGS
import ast
import operator

@tool
def web_search(query: str) -> str:
    """Search the web for current information."""
    results = DDGS().text(query, max_results=3)
    return "\n".join(
        f"{i+1}. {r['title']}: {r['body']}"
        for i, r in enumerate(results)
    )

@tool
def calculator(expression: str) -> str:
    """Calculate a basic mathematical expression."""
    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg
    }

    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value

        if isinstance(node, ast.BinOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](
                evaluate(node.left),
                evaluate(node.right)
            )

        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](
                evaluate(node.operand)
            )

        raise ValueError("Unsupported expression")

    return str(evaluate(ast.parse(expression, mode="eval").body))


tools = [web_search, calculator]

chat_llm = ChatOllama(model="qwen2.5:3b")

agent = create_agent(
    model=chat_llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. "
        "Use web_search for current information and "
        "calculator for mathematical calculations."
    )
)

In [4]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the current population of India?"
        }
    ]
})

print(result["messages"][-1].content)

Based on the information from different sources, the current population of India is approximately 1,494,138,981 as of Tuesday, September 29, 2026. This information is from the site Countrymeters, which provides the most recent and accurate data available.


In [5]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Calculate 1250 * 48 + 750"
        }
    ]
})

print(result["messages"][-1].content)

The result of the calculation 1250 * 48 + 750 is 60750.


In [7]:
population = 1494138981
one_percent = calculator.invoke({"expression": f"{population} * 0.01"})

print(f"Population: {population:,}")
print(f"1% of population: {float(one_percent):,.1f}")

Population: 1,494,138,981
1% of population: 14,941,389.8
